In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# 生成统计图表
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('EdUHK Dataset Statistics', fontsize=16, fontweight='bold')

# 1. 数据集分布饼图
try:
    sizes = [stats.get('trainA', 0), stats.get('trainB', 0), 
             stats.get('testA', 0), stats.get('testB', 0)]
    labels = ['Rain Train (A)', 'Sunny Train (B)', 'Rain Test (A)', 'Sunny Test (B)']
    colors = ['#3498db', '#f39c12', '#2980b9', '#e67e22']
    
    ax1.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
    ax1.set_title('Dataset Distribution')
except:
    ax1.text(0.5, 0.5, 'No data available', ha='center', va='center')

# 2. 各类别图像数量柱状图
try:
    dirs = list(stats.keys())
    values = list(stats.values())
    colors_bar = ['#3498db', '#f39c12', '#2980b9', '#e67e22']
    
    bars = ax2.bar(dirs, values, color=colors_bar)
    ax2.set_ylabel('Number of Images')
    ax2.set_title('Images per Category')
    ax2.axhline(y=150, color='r', linestyle='--', label='Min requirement (150)')
    ax2.legend()
    
    # 在柱子上显示数值
    for bar in bars:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height)}', ha='center', va='bottom')
except:
    ax2.text(0.5, 0.5, 'No data available', ha='center', va='center')

# 3. 训练/测试比例
try:
    train_total = stats.get('trainA', 0) + stats.get('trainB', 0)
    test_total = stats.get('testA', 0) + stats.get('testB', 0)
    
    ax3.bar(['Training', 'Testing'], [train_total, test_total], 
           color=['#2ecc71', '#e74c3c'])
    ax3.set_ylabel('Number of Images')
    ax3.set_title('Training vs Testing Split')
    
    for i, v in enumerate([train_total, test_total]):
        ax3.text(i, v + 10, str(v), ha='center', va='bottom', fontweight='bold')
except:
    ax3.text(0.5, 0.5, 'No data available', ha='center', va='center')

# 4. 统计摘要
summary_text = f"""
Dataset Summary

Total Images: {total_images}
  - Training: {stats.get('trainA', 0) + stats.get('trainB', 0)}
  - Testing: {stats.get('testA', 0) + stats.get('testB', 0)}

Rain Images:
  - Train (A): {stats.get('trainA', 0)}
  - Test (A): {stats.get('testA', 0)}

Sunny Images:
  - Train (B): {stats.get('trainB', 0)}
  - Test (B): {stats.get('testB', 0)}

Status: {"✅ Ready" if total_images >= 300 else "⚠️ Needs more images"}
"""

ax4.text(0.1, 0.95, summary_text, transform=ax4.transAxes,
        fontsize=11, verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
ax4.axis('off')

plt.tight_layout()
plt.show()

print("✅ 统计信息已显示")

---

## 📚 下一步

### 选项1: 开始训练

```bash
# 使用上面生成的命令进行训练
python train.py --dataroot ./datasets/edhuk_weather --name edhuk_rain2sunny ...
```

### 选项2: 查看数据集

```bash
# 浏览数据集图像
open datasets/edhuk_weather/trainA  # macOS
# 或
nautilus datasets/edhuk_weather/trainA  # Linux
```

### 选项3: 获取更多帮助

详见 [EDHUK_DATASET.md](./EDHUK_DATASET.md) 获取完整文档

---

## 📊 数据集统计

您可以运行下一个单元格查看详细的数据集统计信息。

In [ ]:
# 创建训练配置
import json

print("\n⚙️  创建训练配置")
print("=" * 70)

training_configs = {
    'quick_test': {
        'name': 'edhuk_rain2sunny_test',
        'description': '快速测试 (5个周期，单GPU)',
        'n_epochs': 5,
        'n_epochs_decay': 5,
        'batch_size': 1,
        'netG': 'resnet_6blocks',
        'gpu_ids': '0',
        'estimated_time': '30分钟',
    },
    'standard': {
        'name': 'edhuk_rain2sunny',
        'description': '标准训练 (200个周期)',
        'n_epochs': 200,
        'n_epochs_decay': 200,
        'batch_size': 4,
        'netG': 'resnet_9blocks',
        'gpu_ids': '0',
        'estimated_time': '4-6小时 (GPU)',
    },
    'multi_gpu': {
        'name': 'edhuk_rain2sunny_ddp',
        'description': '多GPU训练 (DDP) - 4个GPU',
        'n_epochs': 200,
        'n_epochs_decay': 200,
        'batch_size': 8,
        'netG': 'resnet_9blocks',
        'gpu_ids': '0,1,2,3',
        'estimated_time': '1-2小时',
    }
}

print("\n📋 可用的训练配置:")
print("-" * 70)

for i, (key, config) in enumerate(training_configs.items(), 1):
    print(f"\n{i}. {key.upper()}")
    print(f"   说明: {config['description']}")
    print(f"   周期: {config['n_epochs']}")
    print(f"   批大小: {config['batch_size']}")
    print(f"   预计时间: {config['estimated_time']}")

# 用户选择
print("\n" + "=" * 70)
choice = input("\n选择配置 (输入 'quick_test', 'standard' 或 'multi_gpu'): ").strip().lower()

if choice not in training_configs:
    choice = 'standard'
    print(f"⚠️  无效选择，使用默认: {choice}")

selected_config = training_configs[choice]
config_name = selected_config['name']

# 保存配置
config_file = dataset_root.parent / f"{config_name}_config.json"
config_data = {
    'dataroot': str(dataset_root),
    'name': config_name,
    'model': 'cycle_gan',
    'n_epochs': selected_config['n_epochs'],
    'n_epochs_decay': selected_config['n_epochs_decay'],
    'batch_size': selected_config['batch_size'],
    'netG': selected_config['netG'],
    'netD': 'n_layers',
    'lr': 0.0002,
    'beta1': 0.5,
    'display_freq': 100,
    'print_freq': 100,
    'save_latest_freq': 5000,
    'save_epoch_freq': 1,
    'gpu_ids': selected_config['gpu_ids'],
    'no_dropout': False,
}

with open(config_file, 'w') as f:
    json.dump(config_data, f, indent=2)

print(f"\n✅ 配置已保存: {config_file}")
print(f"\n📝 训练命令:")
print("-" * 70)

# 生成训练命令
cmd_parts = ['python train.py', f'--dataroot {config_data["dataroot"]}']
for key, value in config_data.items():
    if key != 'dataroot':
        cmd_parts.append(f'--{key} {value}')

train_cmd = ' \\\n  '.join(cmd_parts)
print(f"\n{train_cmd}\n")

print("-" * 70)
print(f"✨ 数据集准备完毕！")
print(f"   下一步: 运行上述训练命令")

---

## 第4步: 创建训练配置

In [ ]:
# 验证数据集
print("\n📋 数据集验证")
print("=" * 70)

stats = {}
total_images = 0

for dirname in ['trainA', 'trainB', 'testA', 'testB']:
    dirpath = dataset_root / dirname
    files = list(dirpath.glob('*.jpg')) + list(dirpath.glob('*.png'))
    stats[dirname] = len(files)
    total_images += len(files)
    
    status = "✅" if len(files) > 0 else "❌"
    print(f"  {status} {dirname:10} : {len(files):4} 张图像")

print("-" * 70)
print(f"  📊 总计: {total_images:4} 张图像")

# 检查最小要求
print("\n📋 最小要求检查:")
print("-" * 70)

checks = [
    (stats['trainA'] >= 150, f"trainA >= 150 张 (当前 {stats['trainA']})"),
    (stats['trainB'] >= 150, f"trainB >= 150 张 (当前 {stats['trainB']})"),
    (total_images >= 300, f"总计 >= 300 张 (当前 {total_images})"),
]

for passed, check in checks:
    status = "✅" if passed else "❌"
    print(f"  {status} {check}")

# 检查图像质量
print("\n📊 图像质量检查:")
print("-" * 70)

try:
    from PIL import Image
    import numpy as np
    
    for dirname in ['trainA', 'trainB']:
        dirpath = dataset_root / dirname
        files = list(dirpath.glob('*.jpg')) + list(dirpath.glob('*.png'))
        
        if files:
            sizes = []
            file_sizes = []
            
            for img_file in files[:min(10, len(files))]:  # 检查前10张
                try:
                    img = Image.open(img_file)
                    sizes.append(img.size)
                    file_sizes.append(os.path.getsize(img_file) / 1024)  # KB
                except Exception as e:
                    print(f"  ⚠️  无法读取: {img_file.name}")
            
            if sizes:
                avg_width = int(sum(s[0] for s in sizes) / len(sizes))
                avg_height = int(sum(s[1] for s in sizes) / len(sizes))
                avg_file_size = sum(file_sizes) / len(file_sizes)
                
                size_ok = avg_width >= 256 and avg_height >= 256
                status = "✅" if size_ok else "⚠️"
                
                print(f"  {status} {dirname:10} - 平均分辨率: {avg_width}x{avg_height}, 文件大小: {avg_file_size:.0f}KB")
                
except ImportError:
    print("  ℹ️  安装PIL以检查图像质量: pip install Pillow")

print("\n✅ 验证完成")

---

## 第3步: 验证数据集

---

### 选项 B: 手动下载

如果自动下载失败，可以手动下载。**步骤:**

1. 访问 https://unsplash.com/search/photos?query=rain
2. 搜索 "rain rainy weather" 
3. 下载约250张图像 (JPG格式)
4. 保存到 `datasets/edhuk_weather/trainA/`

重复以上步骤，将晴天图像保存到 `datasets/edhuk_weather/trainB/`

然后运行下面的代码验证。

In [ ]:
from prepare_edhuk_dataset import DatasetCollector

# 初始化收集器
collector = DatasetCollector(str(dataset_root))

# 配置
NUM_IMAGES = 250  # 每种天气类型的图像数
RAIN_QUERIES = [
    "Hong Kong rain rainy weather",
    "university campus rainy day",
    "city rain photography",
]
SUNNY_QUERIES = [
    "Hong Kong sunny clear weather", 
    "university campus sunny day",
    "city sunny photography",
]

print("\n📥 Bing Image Search 下载")
print("=" * 70)

# 显示下载估计
total_images = NUM_IMAGES * 2  # 雨天 + 晴天
print(f"预计下载: {total_images} 张图像 (约{NUM_IMAGES}张雨天 + {NUM_IMAGES}张晴天)")
print(f"预计时间: 10-30 分钟 (取决于网络速度)")
print(f"存储空间: 约 {total_images * 2 / 1024:.1f} MB")

# 用户确认
response = input("\n确认开始下载? (输入 'yes' 确认): ").strip().lower()
if response == 'yes':
    # 下载雨天图像
    print(f"\n📥 下载雨天图像到 trainA/ ...")
    collector.download_from_bing(
        "Hong Kong rain rainy weather",
        num_images=NUM_IMAGES,
        save_dir='trainA'
    )
    
    # 下载晴天图像
    print(f"\n📥 下载晴天图像到 trainB/ ...")
    collector.download_from_bing(
        "Hong Kong sunny clear weather",
        num_images=NUM_IMAGES,
        save_dir='trainB'
    )
    
    print("\n✅ 下载完成!")
else:
    print("⏭️  跳过下载")

---

## 第2步: 下载或添加图像

### 选项 A: 自动下载 (Bing Image Search) - 推荐

运行下面的代码自动下载图像。**首先需要安装依赖:**

```bash
pip install bing-image-downloader requests pillow
```

In [ ]:
# 创建目录结构
dirs_to_create = ['trainA', 'trainB', 'testA', 'testB']

print("\n🔧 创建目录结构...")
print("-" * 70)

for dirname in dirs_to_create:
    dir_path = dataset_root / dirname
    dir_path.mkdir(parents=True, exist_ok=True)
    count = len(list(dir_path.glob('*.jpg'))) + len(list(dir_path.glob('*.png')))
    print(f"  ✅ {dirname:10} - {count:3} 张图像")

print("\n✅ 目录结构准备完成")

---

## 第1步: 创建目录结构

In [ ]:
import os
import sys
import json
from pathlib import Path

# 添加项目路径
sys.path.insert(0, '/Users/leon/Desktop/cycleGAN')

# 创建数据集储存的基础目录
dataset_root = Path('/Users/leon/Desktop/cycleGAN/datasets/edhuk_weather')

print("=" * 70)
print("📸 EdUHK 雨天/晴天数据集准备工具")
print("=" * 70)
print(f"\n📂 数据集根目录: {dataset_root}")
print(f"   状态: {'✅ 存在' if dataset_root.exists() else '❌ 不存在'}")

# 🎯 EdUHK 雨天/晴天数据集准备

## 香港教育大学天气转换CycleGAN项目

本notebook将指导您完成以下步骤：
1. 创建数据集目录结构
2. 自动下载或手动添加图像
3. 验证数据集完整性
4. 准备训练配置